In [2]:
import pandas as pd
import nfl_data_py as nfl

In [3]:
years = list(range(2014, 2025))
weekly = nfl.import_weekly_data(years)
print(weekly.columns)
print(weekly['opponent_team'].unique())
games = nfl.import_schedules(years)
print(games['home_team'].unique())
#'STL' -> 'LA'
#'OAK' -> 'LV'
#'SD' -> 'LAC'
games['home_team'] = games['home_team'].replace({'STL':'LA', 'OAK':'LV', 'SD':'LAC'})
games['away_team'] = games['away_team'].replace({'STL':'LA', 'OAK':'LV', 'SD':'LAC'})
print(games['home_team'].unique())

Downcasting floats.
Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_t

In [4]:
weekly = weekly[((weekly['season'] <= 2020) & (weekly['week'] <= 17)) | ((weekly['season'] >= 2021) & (weekly['week'] <= 18))]
weekly['opponent_team'] = weekly['opponent_team'].replace({'STL':'LA', 'OAK':'LV', 'SD':'LAC'})
print(weekly.columns)
weekly_fil = weekly[['player_id', 'recent_team', 'player_display_name', 'position', 'season', 'week', 'opponent_team', 
          'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions', 'sack_fumbles_lost', 
          'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
          'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles_lost']]
weekly_fil['fumbles_lost'] = (weekly_fil['sack_fumbles_lost']+weekly_fil['rushing_fumbles_lost']+weekly_fil['receiving_fumbles_lost']).fillna(0)
weekly_fil = weekly_fil.drop(columns=['sack_fumbles_lost', 'rushing_fumbles_lost', 'receiving_fumbles_lost'])
weekly_fil.head()

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

C:\Users\rille\AppData\Local\Temp\ipykernel_23768\885754522.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  weekly_fil['fumbles_lost'] = (weekly_fil['sack_fumbles_lost']+weekly_fil['rushing_fumbles_lost']+weekly_fil['receiving_fumbles_lost']).fillna(0)


,player_id,recent_team,player_display_name,position,season,week,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,carries,rushing_yards,rushing_tds,receptions,targets,receiving_yards,receiving_tds,fumbles_lost
0,00-0007091,IND,Matt Hasselbeck,QB,2014,3,JAX,2,4,20.0,0,0.0,1,-1.0,0,0,0,0.0,0,0.0
1,00-0007091,IND,Matt Hasselbeck,QB,2014,4,TEN,0,0,0.0,0,0.0,3,-2.0,0,0,0,0.0,0,0.0
2,00-0007091,IND,Matt Hasselbeck,QB,2014,16,DAL,15,21,126.0,1,0.0,0,0.0,0,0,0,0.0,0,1.0
3,00-0007091,IND,Matt Hasselbeck,QB,2014,17,TEN,13,19,155.0,1,0.0,4,-8.0,0,0,0,0.0,0,0.0
4,00-0010346,DEN,Peyton Manning,QB,2014,1,IND,22,36,269.0,3,0.0,4,-3.0,0,0,0,0.0,0,0.0


In [5]:
def get_game_info(row, df_games):
    year = row["season"]
    week = row["week"]
    opponent = row["opponent_team"]

    candidates = df_games[(df_games["season"] == year) & (df_games["week"] == week)]
    match = candidates[
        (candidates["away_team"] == opponent) | (candidates["home_team"] == opponent)
    ]
    
    if not match.empty:
        if len(match) > 1:
            print("Not only one match")

        game_id = match.iloc[0]["game_id"]
        
        # Si el oponente es el away, el balón lo tiene el home
        if match.iloc[0]["away_team"] == opponent:
            ball_team = match.iloc[0]["home_team"]
        else:
            ball_team = match.iloc[0]["away_team"]
        
        return pd.Series([game_id, ball_team], index=["game_id", "ball_team"])
    else:
        print(f"Did not find game_id for year {year}, week {week}, opponent {opponent}")
        return pd.Series([None, None], index=["game_id", "ball_team"])

print(min(games['season']))
# Aplicación
weekly_fil[["game_id", "ball_team"]] = weekly_fil.apply(
    lambda row: get_game_info(row, games),
    axis=1
)
weekly_fil["rn"] = (
    weekly_fil
    .groupby("player_id")["game_id"]
    .rank(method="first", ascending=True)
    .astype(int)
)
#weekly_fil[['game_id', 'ball_team', 'opponent_team']].head()
#stafford: '00-0026498'
#daniels: '00-0039910'
daniels = weekly_fil[weekly_fil['player_id']=='00-0026498'].sort_values(by='rn')
daniels.head()

2014


,player_id,recent_team,player_display_name,position,season,week,opponent_team,completions,attempts,passing_yards,...,rushing_yards,rushing_tds,receptions,targets,receiving_yards,receiving_tds,fumbles_lost,game_id,ball_team,rn
1439,00-0026498,DET,Matthew Stafford,QB,2014,1,NYG,22,32,346.0,...,2.0,1,0,0,0.0,0,0.0,2014_01_NYG_DET,DET,1
1440,00-0026498,DET,Matthew Stafford,QB,2014,2,CAR,27,48,291.0,...,8.0,0,0,0,0.0,0,0.0,2014_02_DET_CAR,DET,2
1441,00-0026498,DET,Matthew Stafford,QB,2014,3,GB,22,34,246.0,...,8.0,0,0,0,0.0,0,1.0,2014_03_GB_DET,DET,3
1442,00-0026498,DET,Matthew Stafford,QB,2014,4,NYJ,24,34,293.0,...,8.0,1,0,0,0.0,0,0.0,2014_04_DET_NYJ,DET,4
1443,00-0026498,DET,Matthew Stafford,QB,2014,5,BUF,18,31,231.0,...,8.0,0,0,0,0.0,0,0.0,2014_05_BUF_DET,DET,5


In [ ]:
# Agrupar por game_id y ball_team
agrupado = weekly_fil.groupby(['game_id', 'ball_team']).agg({
    'receiving_yards': 'sum',
    'targets': 'sum', 
    'receptions': 'sum',
    'receiving_tds': 'sum',
    'rushing_yards': 'sum',
    'carries': 'sum',
    'rushing_tds': 'sum',
    'passing_yards': 'sum',
    'attempts': 'sum',
    'completions': 'sum',
    'passing_tds': 'sum'
})
agrupado["rn"] = (
    agrupado
    .groupby("ball_team", group_keys=False)
    .apply(lambda x: pd.Series(range(1, len(x) + 1), index=x.index))
)
agrupado.head()

receiving_yards  targets  receptions  \
game_id         ball_team                                         
2014_01_BUF_CHI BUF                  173.0       22          16   
                CHI                  349.0       49          34   
2014_01_CAR_TB  CAR                  230.0       32          24   
                TB                   183.0       35          22   
2014_01_CIN_BAL BAL                  345.0       62          35   

                           receiving_tds  rushing_yards  carries  rushing_tds  \
game_id         ball_team                                                       
2014_01_BUF_CHI BUF                    1          193.0       33            1   
                CHI                    2           86.0       18            0   
2014_01_CAR_TB  CAR                    2          113.0       33            0   
                TB                     2          102.0       17            0   
2014_01_CIN_BAL BAL                    1           94.0       20            1   

                           passing_yards  attempts  completions  passing_tds  \
game_id         ball_team                                                      
2014_01_BUF_CHI BUF                173.0        22           16            1   
                CHI                349.0        49           34            2   
2014_01_CAR_TB  CAR                230.0        34           24            2   
                TB                 183.0        35           22            2   
2014_01_CIN_BAL BAL                345.0        62           35            1   

                           rn  
game_id         ball_team      
2014_01_BUF_CHI BUF         1  
                CHI         1  
2014_01_CAR_TB  CAR         1  
                TB          1  
2014_01_CIN_BAL BAL         1

In [12]:
# Agrupar por game_id y opponent_team
agrupado_opp = weekly_fil.groupby(['game_id', 'opponent_team']).agg({
    'receiving_yards': 'sum', 
    'receiving_tds': 'sum',
    'passing_yards': 'sum',
    'passing_tds': 'sum',
    'interceptions': 'sum',
    'rushing_yards':'sum',
    'rushing_tds': 'sum',
    'fumbles_lost': 'sum'
})
agrupado_opp["rn"] = (
    agrupado_opp
    .groupby("opponent_team", group_keys=False)
    .apply(lambda x: pd.Series(range(1, len(x) + 1), index=x.index))
)
agrupado_opp.head()

receiving_yards  receiving_tds  passing_yards  \
game_id         opponent_team                                                  
2014_01_BUF_CHI BUF                      349.0              2          349.0   
                CHI                      173.0              1          173.0   
2014_01_CAR_TB  CAR                      183.0              2          183.0   
                TB                       230.0              2          230.0   
2014_01_CIN_BAL BAL                      301.0              1          301.0   

                               passing_tds  interceptions  rushing_yards  \
game_id         opponent_team                                              
2014_01_BUF_CHI BUF                      2            2.0           86.0   
                CHI                      1            1.0          193.0   
2014_01_CAR_TB  CAR                      2            2.0          102.0   
                TB                       2            0.0          113.0   
2014_01_CIN_BAL BAL                      1            0.0           79.0   

                               rushing_tds  fumbles_lost  rn  
game_id         opponent_team                                 
2014_01_BUF_CHI BUF                      0           1.0   1  
                CHI                      1           0.0   1  
2014_01_CAR_TB  CAR                      0           1.0   1  
                TB                       0           0.0   1  
2014_01_CIN_BAL BAL                      0           0.0   1

In [13]:
weekly_fil.to_csv('player_weekly_data_2014_24.csv', index=False)
agrupado.to_csv('team_offense_2014_24.csv')
agrupado_opp.to_csv('team_defense_2014_24.csv')